In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [3]:
data = pd.read_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/features_temporal.csv")

In [5]:
data.head()

,subject,age,sex,test_time,motor_updrs,total_updrs,jitter,jitter_abs,jitter_rap,jitter_ppq5,...,nhr,hnr,rpde,dfa,ppe,motor_updrs_lag1,delta_time,delta_updrs,updrs_rolling_mean_3,updrs_rolling_std_3
0,1,72,False,5.6438,28.199,34.398,0.00413,0.000021,0.00173,0.00165,...,0.030790,26.641,0.50911,0.53637,0.251830,28.199,0.0007,0.000,28.199000,0.000000
1,1,72,False,5.6451,28.199,34.398,0.00217,0.000011,0.00086,0.00099,...,0.004547,30.749,0.41216,0.54572,0.094704,28.199,0.0013,0.000,28.199000,0.000000
2,1,72,False,5.6458,28.199,34.399,0.00294,0.000015,0.00121,0.00125,...,0.014919,27.530,0.40986,0.53572,0.166350,28.199,0.0007,0.000,28.199000,0.000000
3,1,72,False,5.6458,28.199,34.399,0.00250,0.000013,0.00110,0.00090,...,0.003822,32.102,0.31053,0.52767,0.065937,28.199,0.0000,0.000,28.199000,0.000000
4,1,72,False,12.6660,28.447,34.894,0.00300,0.000017,0.00132,0.00150,...,0.011112,27.183,0.43493,0.56477,0.108100,28.199,7.0202,0.248,28.281667,0.143183


In [9]:
assert "subject" in data.columns
assert "motor_updrs" in data.columns


In [12]:
feature_cols = [
    "jitter", "shimmer", "hnr",
    "dfa", "rpde", "ppe",
    "age", "sex",
    "motor_updrs_lag1",
    "delta_time",
    "delta_updrs",
    "updrs_rolling_mean_3",
    "updrs_rolling_std_3"
]

X = data[feature_cols]
y = data["motor_updrs"]
groups = data["subject"]


In [14]:
gkf = GroupKFold(n_splits=5)

In [16]:
models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        random_state=42
    ),
    "SVR": SVR(
        kernel="rbf",
        C=10,
        epsilon=0.5
    )
}

In [21]:
results = []

for model_name, model in models.items():
    fold_mae = []
    fold_rmse = []
    fold_r2 = []

    for fold, (train_idx, test_idx) in enumerate(
        gkf.split(X, y, groups=groups)
    ):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        fold_mae.append(mean_absolute_error(y_test, y_pred))
        fold_rmse.append(mean_squared_error(y_test, y_pred**0.5))
        fold_r2.append(r2_score(y_test, y_pred))

    results.append({
        "model": model_name,
        "MAE": np.mean(fold_mae),
        "RMSE": np.mean(fold_rmse),
        "R2": np.mean(fold_r2)
    })


In [24]:
results_data = pd.DataFrame(results).sort_values("MAE")
results_data

,model,MAE,RMSE,R2
1,Gradient Boosting,0.072012,335.816264,0.999656
0,Random Forest,0.102454,335.813677,0.999349
2,SVR,0.274479,335.928745,0.996073


In [26]:
best_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

best_model.fit(X, y)
data["pred_motor_updrs"] = best_model.predict(X)

data.to_csv("C:/Users/Durosimi/PROJECTS/Parkinson/data/processed/predictions_temporal.csv", index=False)


In [28]:
data.groupby("subject")[["motor_updrs", "pred_motor_updrs"]].corr()

motor_updrs  pred_motor_updrs
subject                                                
1       motor_updrs          1.000000          0.999757
        pred_motor_updrs     0.999757          1.000000
2       motor_updrs          1.000000          0.999531
        pred_motor_updrs     0.999531          1.000000
3       motor_updrs          1.000000          0.999655
...                               ...               ...
40      pred_motor_updrs     0.998012          1.000000
41      motor_updrs          1.000000          0.998843
        pred_motor_updrs     0.998843          1.000000
42      motor_updrs          1.000000          0.998921
        pred_motor_updrs     0.998921          1.000000

[84 rows x 2 columns]